In [19]:
import os
from pathlib import Path
from glob import glob
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Prepare and Store Vectors

In [2]:
base_path = Path("knowledge-base")
documents = []

for folder in base_path.iterdir():
    doc_type = folder.name
    loader = DirectoryLoader(folder, 
                            glob="**/*.md",
                            loader_cls=TextLoader, 
                            loader_kwargs={'encoding':'utf-8'}, 
                            show_progress=True)
    docs = loader.load()
    for doc in docs:
        doc.metadata['doc_type'] = doc_type
    
    documents.extend(docs)


100%|██████████| 8/8 [00:00<00:00, 4694.24it/s]


In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

len(chunks)

413

In [12]:
store_name = 'vector_db_insurellm'
embeddings = HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

if os.path.exists(store_name):
    Chroma(embedding_function=embeddings, persist_directory=store_name).delete_collection()

vectorstore = Chroma.from_documents(chunks, embedding=embeddings, persist_directory=store_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Read Vectors from Store

In [14]:
vectors = Chroma(
    persist_directory=store_name,
    embedding_function=embeddings
)

# Build the Pipeline

In [21]:
MODEL = "gpt-4.1-nano"
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

In [22]:
llm = ChatOpenAI(model_name=MODEL, temperature=0)
retriever = vectors.as_retriever()

In [23]:
retriever.invoke("Who is Avery?")

[Document(id='9fbaa667-95ef-4fa3-8418-a45fb5d6aa0d', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [ ]:
# Not RAG yet
llm.invoke("Who is Avery?")

AIMessage(content="Avery is a given name that can be used for both males and females. It can also be a surname. Without additional context, it's difficult to determine which specific Avery you're referring to. If you can provide more details—such as a full name, profession, or context—I’d be happy to help you with more specific information.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 11, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_8ea04f4c59', 'id': 'chatcmpl-E00mNnWObmZLisv4d92lpFT7QDcQh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f4b1e-3caf-7d83-9b4f-f043df184d57-0', tool_calls=[], invalid_tool_cal

In [27]:
SYSTEM_PROMPT_TEMPLATE = """You are an expert on InsureLM, a company that provides insurance related products.
You are able to answer questions related to InsureLM, its products and employees. 
You are provided additional context that might be relevant to a user's question. 
Give brief, accurate answers.
If you don't know the answer to any question, just say so.
Respond in less than 100 words.

Context:
{context}
"""

In [33]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    # print(f"Additional Context: \n\n {context}")
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])
    return response.content

In [34]:
answer_question("Who is Avery?", [])

"Avery Lancaster is the Co-Founder and CEO of Insurellm, a leading insurance technology company based in San Francisco. She has been instrumental in guiding the company's growth and innovation in the insurance industry."

In [35]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
